# Assembling Java Byte Code with *Pytunia*

The *Java* virtual machine (JVM) executes *class files*.  These files contain *Java byte code*, which is the machine
code of the JVM.  Usually, class files are created by the *Java* compiler `javac`.  In this notebook we write the
byte code ourselves:  The files `Hello.jas` and `Sum.jas` contain programs written in *JVM assembler*, a readable
notation for *Java* byte code.  The *Python* package [Pytunia](https://github.com/karlstroetmann/pytunia)
translates these files into class files.  *Pytunia* is derived from the assembler
[Jasmin](https://jasmin.sourceforge.net), but its syntax has been updated:  For example, class names and types are
written as in *Java*.  Finally, we execute these class files with the command `java`.

*Pytunia* is installed with the command
```
pip install pytunia
```

The class `Code` from the module `IPython.display` displays the contents of a file with syntax highlighting.  The
highlighting is done by the library [Pygments](https://pygments.org), whose lexer `jasmin` also highlights the
assembler code of *Pytunia*.

In [ ]:
from IPython.display import Code
import pytunia

## Hello World

The file `Hello.jas` contains a program that prints the string `Hello World!`.

In [ ]:
Code(filename='Hello.jas', language='jasmin')

This program is discussed in detail in chapter 12 of the lecture notes.  In short:
- The directive `.class` declares the class `Hello`, and the directive `.super` declares its superclass
  `java.lang.Object`.  Classes are always specified by their full names.
- The first method, `<init>`, is the constructor of the class `Hello`.  It calls the constructor of the class
  `java.lang.Object`.
- The second method is the method `main`.  Its signature `main(java.lang.String[]) void` is written with the
  types of *Java*:  `main` takes an array of strings as its argument and returns nothing.
  - `.limit locals 1` reserves one local variable, which holds the argument of `main`, and `.limit stack 2`
    specifies that the stack holds at most two values.
  - `getstatic` pushes the object `java.lang.System.out`, which is of the class `java.io.PrintStream`, onto the
    stack.  Then `ldc` pushes the string `"Hello World!"`.
  - `invokevirtual` calls the method `println` of this object with the string as its argument.  Both values are
    removed from the stack.
  - `return` returns from the method `main`.

The function `pytunia.assemble` translates the assembler file into the class file `Hello.class` and returns the path
of this file.  If the assembler file contained an error, `pytunia.assemble` would raise an exception of the class
`pytunia.AssemblyError` instead, and the message of this exception would describe the error.

In [ ]:
pytunia.assemble('Hello.jas')

In [ ]:
!ls -l

A class file contains binary data.  Every class file starts with the four bytes `CA FE BA BE`, which are known as
the *magic number* of class files.

In [ ]:
with open('Hello.class', 'rb') as handle:
    print(handle.read(4).hex())

The command `java` starts the *Java* virtual machine.  It loads the class `Hello` from the file `Hello.class` in the
current directory and calls the method `main` of this class.  In a notebook, a line that starts with the character
`!` is executed as a shell command.  (If the environment variable `CLASSPATH` is set, it has to contain the current
directory `.`, as otherwise the class `Hello` is not found.)

In [ ]:
!java Hello

## Computing a Sum

The file `Sum.jas` contains a program that computes the sum 
$$ \sum\limits_{i=1}^{n} i $$ 
for the value $n = 6 \cdot 6$ and prints the result.  The following *Java* program does the same computation.

In [ ]:
Code(filename='Sum.java', language='java')

In [ ]:
Code(filename='Sum.jas', language='jasmin')

- The method `main` first pushes the object `java.lang.System.out` onto the stack, since this object is needed later
  for calling `println`.  Then `ldc 6` pushes the number $6$, `dup` duplicates it, and `imul` replaces both copies
  by their product $36$.  Next, `invokestatic` calls the static method `sum` of the class `Sum`.  Its signature
  `Sum.sum(int) int` states that `sum` takes an argument of type `int` and returns an `int`.  On the stack, the
  argument $36$ is replaced by the result of `sum`.  Finally, `invokevirtual` prints this result.
- In the method `sum`, the local variable $0$ holds the argument $n$ and the local variable $1$ holds the sum $s$.
  In contrast to the *Java* program, the loop counts $n$ down to $0$:  As long as $n \not= 0$, it adds $n$ to $s$
  and decrements $n$.  The instruction `ifeq finish` leaves the loop as soon as $n = 0$, while `goto loop` jumps
  back to the beginning of the loop.  Finally, `ireturn` returns the value of $s$.

We assemble the file `Sum.jas` and execute the resulting class file.  The output should be
$$ \sum\limits_{i=1}^{36} i = \frac{36 \cdot 37}{2} = 666. $$

In [ ]:
pytunia.assemble('Sum.jas')

In [ ]:
!java Sum